In [ ]:
import numpy as np
import time
from algos.naive import naive_search
from algos.rabin_karp import rabin_karp_search
from algos.greedy import greedy_kmp_search, greedy_bm_search

import matplotlib.pyplot as plt
import pandas as pd

def get_hollow_pattern(size):
    p = np.zeros((size, size), dtype=int)
    p[0, :] = 1; p[-1, :] = 1; p[:, 0] = 1; p[:, -1] = 1
    return p

def run_suite():
    dims = [100, 500, 1000, 5000] # Scaled for reasonable completion time
    # We will do 10000 separately or at the end for specific stats
    p_sizes = [3, 5, 7]
    algos = [
        ("Naive", naive_search),
        ("Rabin-Karp", rabin_karp_search),
        ("Greedy KMP", greedy_kmp_search),
        ("Greedy BM", greedy_bm_search)
    ]
    
    results = []
    
    # 1. Random Tests
    for dim in dims:
        for ps in p_sizes:
            np.random.seed(62)
            text = np.random.randint(2, size=(dim, dim))
            # Ensure no match early for growth curve (or just random)
            # Actually, randomized is fine.
            pattern = get_hollow_pattern(ps)
            for name, func in algos:
                start = time.perf_counter()
                count, pos = func(text, pattern)
                elapsed = time.perf_counter() - start
                results.append({"dim": dim, "p_size": ps, "algo": name, "type": "Random", "comparisons": count, "time": elapsed})
                
    # 2. Contrived "Trigger" Tests
    for dim in dims:
        for ps in p_sizes:
            # Pattern starts with 1s. Text has first row of pattern at start of every row.
            text = np.zeros((dim, dim), dtype=int)
            pattern = get_hollow_pattern(ps)
            text[:, :ps] = pattern[0]
            # Pattern actually exists at the very end
            text[-ps:, -ps:] = pattern
            
            for name, func in algos:
                start = time.perf_counter()
                count, pos = func(text, pattern)
                elapsed = time.perf_counter() - start
                results.append({"dim": dim, "p_size": ps, "algo": name, "type": "Trigger", "comparisons": count, "time": elapsed})

    return pd.DataFrame(results)

df = run_suite()
df.to_csv("benchmark_results.csv", index=False)

# --- 3. Visualizations ---

def plot_results(df):
    types = df['type'].unique()
    p_sizes = df['p_size'].unique()
    
    for t in types:
        for ps in p_sizes:
            subset = df[(df['type'] == t) & (df['p_size'] == ps)]
            
            plt.figure(figsize=(10, 6))
            for algo in subset['algo'].unique():
                algo_data = subset[subset['algo'] == algo]
                plt.plot(algo_data['dim'], algo_data['comparisons'], marker='o', label=algo)
            
            plt.title(f"Comparison Growth: {t} Case (Pattern {ps}x{ps})")
            plt.xlabel("Text Dimension (N of NxN)")
            plt.ylabel("Number of Comparisons")
            plt.grid(True, which="both", ls="-", alpha=0.5)
            plt.legend()
            plt.savefig(f"growth_{t}_p{ps}.png")
            plt.close()

plot_results(df)

KeyboardInterrupt: 

In [ ]:
# Load the data generated in the previous step
df = pd.read_csv("benchmark_results.csv")

# Create a multi-panel figure
# We will create 2 rows (Random vs Trigger) and 3 columns (Pattern sizes 3, 5, 7)
fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharex=True)
fig.suptitle('Ablation Study: Comparison Count Growth (Log-Log Scale)', fontsize=20, fontweight='bold')

test_types = ['Random', 'Trigger']
p_sizes = [3, 5, 7]
colors = {'Naive': '#e74c3c', 'Rabin-Karp': '#9b59b6', 'Greedy KMP': '#3498db', 'Greedy BM': '#2ecc71'}

for i, t_type in enumerate(test_types):
    for j, p_size in enumerate(p_sizes):
        ax = axes[i, j]
        subset = df[(df['type'] == t_type) & (df['p_size'] == p_size)]
        
        for algo in subset['algo'].unique():
            algo_data = subset[subset['algo'] == algo]
            ax.plot(algo_data['dim'], algo_data['comparisons'], 
                    marker='o', label=algo, color=colors.get(algo, None), linewidth=2)
        
        ax.set_yscale('log')
        ax.set_xscale('log')
        ax.grid(True, which="both", ls="--", alpha=0.5)
        
        if i == 0:
            ax.set_title(f'Pattern Size: {p_size}x{p_size}', fontsize=14)
        if j == 0:
            ax.set_ylabel(f'{t_type} Case\nComparisons', fontsize=14, fontweight='bold')
        if i == 1:
            ax.set_xlabel('Matrix Dimension (N)', fontsize=12)

# Add a single legend for the whole figure
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.98, 0.95), fontsize=12)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()